# Chapter 3 — The Reveal: Which Communities Are Being Left Behind?

---

## The Story

Margaret is 84 years old and lives in Central Highlands, Tasmania. Her GP has assessed her as Level 4 — the highest home care need. She qualifies for a residential aged care bed. But there are only 18 beds in her entire region, and 56 people ahead of her with the same urgent need.

**Her waitlist ratio: 3.1 — meaning for every 1 bed, 3 people are waiting.**

Margaret's story is not unusual. Across Australia, 58 SA3 regions are in the same situation: demand structurally exceeds supply. This chapter names them.

---

## Narrative Arc — The Detective: Resolution

Ch 1 showed the map of where the gap is. Ch 2 showed why quality differs (ownership). **Ch 3 is the resolution** — the specific communities where the gap is not just statistical, it is a lived crisis.

**Three questions this chapter answers:**
1. **Where?** — Which 20 SA3 regions have the worst supply–demand mismatch?
2. **Who?** — Who exactly is waiting, and how urgent is their need?
3. **Is it getting worse?** — Year-over-year: are more communities falling into deficit?

---

## User Stories

| Audience | User Story | Acceptance Criteria |
|----------|-----------|--------------------|
| Family caregiver | As a family member planning residential care for an elderly parent, I need to know which regions have the worst waitlists so I can make informed decisions about location | Dashboard shows ranked list with hover detail — specific numbers per region |
| Health strategist | As a policy maker, I need to identify which regions require urgent bed expansion so I can direct infrastructure funding | Top 20 ranked by pressure with state and remoteness — exportable |
| Sector worker | As someone entering the aged care workforce, I need to know where demand is highest so I can find job security | Map of regions in deficit clearly marked, with demand growth trend |

## Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

CLEAN = '../../data/clean'

master = pd.read_csv(f'{CLEAN}/master_sa3.csv')
users  = pd.read_csv(f'{CLEAN}/service_users_by_sa3.csv')

YEAR = 2024
df = master[master['year'] == YEAR].copy()

# Pre-compute key narrative numbers for inline reference
n_over_1  = (df.drop_duplicates('sa3_name')['waitlist_pressure'] > 1.0).sum()
n_over_2  = (df.drop_duplicates('sa3_name')['waitlist_pressure'] > 2.0).sum()
med_wp    = df['waitlist_pressure'].median()
worst_sa3 = df.nlargest(1,'waitlist_pressure').iloc[0]

print(f'Year: {YEAR}')
print(f'Communities where demand exceeds supply (pressure > 1.0): {n_over_1}')
print(f'Communities where demand is DOUBLE supply (pressure > 2.0): {n_over_2}')
print(f'National median pressure: {med_wp:.3f}')
print(f'Worst region: {worst_sa3["sa3_name"]} ({worst_sa3["state"]}) = {worst_sa3["waitlist_pressure"]:.3f}')

Year: 2024
Communities where demand exceeds supply (pressure > 1.0): 58
Communities where demand is DOUBLE supply (pressure > 2.0): 8
National median pressure: 0.636
Worst region: Central Highlands (Tas.) (TAS) = 3.111


### What this tells us

The national picture is stark before we even look at individual communities. **58 SA3 regions** — nearly 1 in 6 of all analysed areas — have more high-needs home care users than available residential beds. **8 of those have double the demand.** The national median of 0.636 means that even in a typical region, for every 10 beds there are already 6 people with high-level needs in home care who could qualify for residential placement. Central Highlands, Tasmania holds the worst ratio in the country at 3.111 — more than three high-needs people for every single available bed.

## What: The Scale of the Problem

Before naming specific communities, we need to understand the national picture.

The home care system was designed to keep people in their homes with support. But it is increasingly being used as a **holding pattern** — people approved for L3 and L4 care (near-residential intensity) who cannot access a residential bed, so they stay at home with ever-increasing care needs.

**What L3 and L4 actually means:**
- **L3 (High):** Requires significant daily assistance — help with bathing, dressing, medication management, meal preparation. Roughly equivalent to 3–4 hours of care per day.
- **L4 (Very High):** Requires near-constant support — dementia care, complex wound management, mobility assistance. These are people who, in most clinical assessments, belong in residential care.

The chart below shows how the composition of home care has shifted since 2023.

In [2]:
nat = users.groupby('year')[[
    'hcp_level1','hcp_level2','hcp_level3','hcp_level4','total_homecare'
]].sum().reset_index()

nat_melt = nat.melt(
    id_vars='year',
    value_vars=['hcp_level1','hcp_level2','hcp_level3','hcp_level4'],
    var_name='level', value_name='users'
)
nat_melt['level_label'] = nat_melt['level'].map({
    'hcp_level1': 'L1 — Basic daily support',
    'hcp_level2': 'L2 — Moderate support',
    'hcp_level3': 'L3 — High (near-residential)',
    'hcp_level4': 'L4 — Very High (should be in residential care)'
})

hcp_2023 = users[users['year']==2023]['hcp_high_needs'].sum()
hcp_2025 = users[users['year']==2025]['hcp_high_needs'].sum()
pct_2025 = users[users['year']==2025]['hcp_high_needs'].sum() / users[users['year']==2025]['total_homecare'].sum() * 100

fig = px.area(
    nat_melt,
    x='year', y='users', color='level_label',
    category_orders={'level_label': [
        'L1 — Basic daily support',
        'L2 — Moderate support',
        'L3 — High (near-residential)',
        'L4 — Very High (should be in residential care)'
    ]},
    color_discrete_map={
        'L1 — Basic daily support':                   '#a6c8ff',
        'L2 — Moderate support':                      '#4589ff',
        'L3 — High (near-residential)':               '#ff832b',
        'L4 — Very High (should be in residential care)': '#da1e28'
    },
    title='Australia\'s home care system is carrying people who need residential care<br>'
          f'<sup>L3+L4 users grew from {hcp_2023:,.0f} (2023) to {hcp_2025:,.0f} (2025) — '
          f'+22% in 2 years. Now {pct_2025:.0f}% of all home care approvals.</sup>',
    labels={'users': 'People receiving home care', 'year': '', 'level_label': 'Care level'}
)
fig.update_layout(height=400)
fig.show()

print(f'Key number: {pct_2025:.1f}% of all home care approvals are now L3 or L4 (2025).')
print(f'These {hcp_2025:,.0f} people need residential care — but the beds are not there.')

Key number: 59.1% of all home care approvals are now L3 or L4 (2025).
These 172,285 people need residential care — but the beds are not there.


### What this chart reveals

The home care system was designed to help people live independently at home with light support. The stacked area chart shows it has become something fundamentally different. **59.1% of all home care approvals in 2025 are now L3 or L4** — the two highest care levels, representing people with complex, intensive needs who in most clinical assessments belong in residential care.

The orange and red layers (L3 and L4) are visibly growing relative to the blue layers (L1 and L2). This is not a temporary spike — it is a structural shift driven by an ageing population accumulating in the home care system because residential beds are not being built fast enough. The 172,285 people at L3+L4 in 2025 represent a +22% increase from 140,968 in just two years.

## So What: These Are the Communities Left Behind

**`waitlist_pressure` = high-needs HCP users (L3+L4) ÷ available residential beds**

A pressure of 1.0 means demand exactly equals supply. Every region above 1.0 is in structural deficit — there are more people who need beds than beds available.

**The reference line at 1.0 is the line between manageable and crisis.**

Colour shows remoteness (MMM). The pattern is striking: this is not purely a remote problem. The majority of the top 20 are MM1 — major cities — where population density creates concentrated demand that supply has not kept pace with.

In [3]:
top20 = (
    df.nlargest(30, 'waitlist_pressure')
    .drop_duplicates('sa3_name')
    .head(20)
    .sort_values('waitlist_pressure')
)

MMM_COLOURS = {
    'MM1':'#1f77b4','MM2':'#ff7f0e','MM3':'#2ca02c',
    'MM4':'#d62728','MM5':'#9467bd','MM6':'#8c564b','MM7':'#e377c2'
}

fig = px.bar(
    top20,
    x='waitlist_pressure', y='sa3_name',
    color='mmm_code',
    color_discrete_map=MMM_COLOURS,
    orientation='h',
    hover_data={
        'state': True,
        'hcp_high_needs': ':,d',
        'residential_places': ':,d',
        'waitlist_pressure': ':.2f',
        'pop_65_plus': ':,.0f'
    },
    title='20 communities where demand has outrun supply — 2024<br>'
          '<sup>Every bar past the red line represents a community in structural deficit. '
          'These are real places where real people cannot get a bed.</sup>',
    labels={
        'waitlist_pressure': 'High-needs home care users per residential bed',
        'sa3_name': '',
        'mmm_code': 'Remoteness'
    }
)
fig.add_vline(
    x=1.0, line_dash='dash', line_color='red', line_width=2,
    annotation_text='Demand = Supply',
    annotation_position='top right'
)
fig.update_layout(height=540)
fig.show()

# So what statement
worst = top20.iloc[-1]
second = top20.iloc[-2]
mm1_count = (top20['mmm_code'] == 'MM1').sum()
print(f'So what: {worst["sa3_name"]} ({worst["state"]}) has {worst["waitlist_pressure"]:.1f} '
      f'high-needs people per bed.')
print(f'  For a family there: finding a residential bed is not difficult — it is effectively impossible.')
print(f'  {second["sa3_name"]} is second at {second["waitlist_pressure"]:.2f} — '
      f'{int(second["hcp_high_needs"])} people, {int(second["residential_places"])} beds.')
print(f'  {mm1_count} of the top 20 are major cities (MM1) — this is not a remote problem.')

So what: Central Highlands (Tas.) (TAS) has 3.1 high-needs people per bed.
  For a family there: finding a residential bed is not difficult — it is effectively impossible.
  Noosa Hinterland is second at 2.83 — 255 people, 90 beds.
  12 of the top 20 are major cities (MM1) — this is not a remote problem.


### What this chart reveals

Every bar that extends past the red dashed line represents a community in structural deficit. The line is not an aspirational target — it is the point where one available bed exists for every high-needs person waiting. Anything beyond it means the system has already failed to keep pace with demand.

**Central Highlands (TAS) at 3.1** is the most extreme case nationally: 56 people competing for 18 beds. For a family in this region, the wait is not measured in weeks — it is effectively indefinite. **Noosa Hinterland (QLD) at 2.83** is second: 255 people, 90 beds.

The colour pattern challenges the assumption that this is a remote problem. **12 of the top 20 are MM1 major cities** — inner and outer suburban areas where population density has created concentrated demand that supply has not followed. The crisis is geographic, not demographic — it is about where beds are, not who needs them.

## Zooming In: Who Is Actually Waiting?

The bar chart above shows *how many* people are waiting relative to supply. This chart answers *who* they are.

In every one of these 20 communities, the dominant colours are orange and red — L3 and L4. These are not people with minor support needs who could be managed with home visits. They are people with complex, intensive care needs who have been assessed as requiring near-residential or residential-level support.

**The orange and red bars are not a queue — they are a crisis being absorbed invisibly into the home care system.**

In [4]:
hcp_melt = top20.melt(
    id_vars=['sa3_name','waitlist_pressure'],
    value_vars=['hcp_level1','hcp_level2','hcp_level3','hcp_level4'],
    var_name='level', value_name='users'
)
hcp_melt['level_label'] = hcp_melt['level'].map({
    'hcp_level1': 'L1 — Basic',
    'hcp_level2': 'L2 — Moderate',
    'hcp_level3': 'L3 — High (near-residential)',
    'hcp_level4': 'L4 — Very High (should be in residential)'
})

sa3_order = top20.sort_values('waitlist_pressure')['sa3_name'].tolist()

fig = px.bar(
    hcp_melt,
    x='sa3_name', y='users', color='level_label',
    category_orders={
        'sa3_name': sa3_order,
        'level_label': ['L1 — Basic','L2 — Moderate',
                        'L3 — High (near-residential)',
                        'L4 — Very High (should be in residential)']
    },
    color_discrete_map={
        'L1 — Basic':                         '#a6c8ff',
        'L2 — Moderate':                      '#4589ff',
        'L3 — High (near-residential)':       '#ff832b',
        'L4 — Very High (should be in residential)': '#da1e28'
    },
    barmode='stack',
    title='The people waiting — care levels in the 20 most pressured communities<br>'
          '<sup>Orange and red = people who need residential care, '
          'not just home support. In many regions this is the majority.</sup>',
    labels={'users':'People receiving home care','sa3_name':'','level_label':'Care level'}
)
fig.update_layout(height=440, xaxis_tickangle=45)
fig.show()

l34 = top20[['hcp_level3','hcp_level4']].sum().sum()
total = top20[['hcp_level1','hcp_level2','hcp_level3','hcp_level4']].sum().sum()
print(f'Across the top 20 communities: {l34/total*100:.1f}% of home care users are L3+L4.')
print(f'That is {int(l34):,} people receiving near-residential care at home')
print(f'because their community does not have enough beds to place them.')

Across the top 20 communities: 65.8% of home care users are L3+L4.
That is 10,086 people receiving near-residential care at home
because their community does not have enough beds to place them.


### What this chart reveals

The stacked bar answers the question the ranked list cannot: *who* is in the queue? In every one of these 20 communities, orange and red — L3 and L4 — dominate the stack. These are not people receiving a weekly shopping assist. They are people with dementia, complex wound care, full mobility dependence, or cognitive impairment severe enough that clinical assessors have categorised them as requiring near-residential-level support.

**Across the top 20 communities, 65.8% of all home care users are at L3+L4.** That is 10,086 people receiving near-residential care at home — against only 5,189 available residential beds, a combined ratio of 1.94x. The queue is not a line of people who would prefer home care. It is a queue of people who have exhausted what home care can safely provide.

## The Details: Every Community, Every Number

For families making a decision, for planners allocating funding, for workers choosing where to work — the ranked list above needs to be paired with the specific numbers.

Two things stand out in this table:

**Access rate and pressure do not always correlate.** Central Highlands TAS has a 0.64% access rate — very few people are actually in residential care — yet it has the worst pressure in Australia. The demand is real, the beds are simply not there. Surfers Paradise QLD is similar: 0.44% access rate, but 2.0 pressure.

**Some high-pressure regions also have lower quality.** The Hills District QLD (pressure 2.029) has a quality score of 2.969 — below standard. Families in these communities face both a waitlist and quality risk if they do get a place.

In [5]:
detail = top20.sort_values('waitlist_pressure', ascending=False)[[
    'sa3_name','state','mmm_code',
    'hcp_high_needs','residential_places','waitlist_pressure',
    'access_rate','quality_score','pop_65_plus'
]].rename(columns={
    'sa3_name':'SA3','state':'State','mmm_code':'MMM',
    'hcp_high_needs':'HCP L3+L4','residential_places':'Beds',
    'waitlist_pressure':'Pressure','access_rate':'Access %',
    'quality_score':'Quality','pop_65_plus':'Pop 65+'
})
detail['Pressure'] = detail['Pressure'].round(3)
detail['Access %'] = detail['Access %'].round(2)
detail['Quality']  = detail['Quality'].round(3)
detail['Pop 65+']  = detail['Pop 65+'].astype(int)
print(detail.reset_index(drop=True).to_string(index=False))

                       SA3 State MMM  HCP L3+L4  Beds  Pressure  Access %  Quality  Pop 65+
  Central Highlands (Tas.)   TAS MM5         56  18.0     3.111      0.64    4.562     2829
          Noosa Hinterland   QLD MM2        255  90.0     2.833      1.23    3.562     6505
 Sunshine Coast Hinterland   QLD MM1        659 252.0     2.615      1.72    3.806    14275
        Wheat Belt - North    WA MM4        964 369.0     2.612      0.89    3.433    14662
         Gympie - Cooloola   QLD MM3       1034 445.0     2.324      2.67    3.562    14930
                     Yarra   VIC MM1        618 270.0     2.289      2.13    4.042    11521
        The Hills District   QLD MM1        345 170.0     2.029      1.11    2.969    14423
                   Kwinana    WA MM1        249 123.0     2.024      2.22    3.188     5185
          Surfers Paradise   QLD MM1        144  72.0     2.000      0.44    3.562    11509
               Carlingford   NSW MM1        399 215.0     1.856      1.70    3.2

### Reading the table

Two patterns emerge that the chart cannot show:

**Low access rate + high pressure = supply failure, not low demand.** Central Highlands TAS has a 0.64% access rate — almost no one is getting into residential care — yet pressure is 3.111. Surfers Paradise QLD is even more extreme: 0.44% access rate, pressure of 2.0. These are not communities where people choose home care over residential. They are communities where residential care is simply unavailable, so people accumulate in home care regardless of their clinical need.

**High pressure + low quality = compounded crisis.** The Hills District QLD (pressure 2.029, quality 2.969) and Kwinana WA (pressure 2.024, quality 3.188) face both problems simultaneously — families cannot get a bed, and the beds that exist are below standard. For these communities, the crisis is not just about quantity. It is about whether the available care is safe.

## What Next: Is It Getting Worse?

The map of crisis communities is not static. Between 2023 and 2024, **10 new SA3 regions crossed the pressure = 1.0 threshold** — communities where demand now structurally exceeds supply for the first time.

This matters because the trend is not cyclical. It is driven by demographics: Australia's 65+ population grew from 4.0M (2019) to 4.7M (2024), and this cohort is aging further into high-dependency years. Supply cannot be built fast enough to keep pace with a demographic wave that was foreseeable decades ago.

**The policy window is now.** Each year of inaction adds more communities to the crisis list.

In [6]:
m23 = master[master['year']==2023].drop_duplicates('sa3_name')
m24 = master[master['year']==2024].drop_duplicates('sa3_name')

# SA3s that crossed 1.0 threshold between 2023 and 2024
over1_2023 = set(m23[m23['waitlist_pressure']>1.0]['sa3_name'])
over1_2024 = set(m24[m24['waitlist_pressure']>1.0]['sa3_name'])
new_crisis  = over1_2024 - over1_2023
resolved    = over1_2023 - over1_2024

print(f'Communities in deficit (pressure > 1.0):')
print(f'  2023: {len(over1_2023)} SA3s')
print(f'  2024: {len(over1_2024)} SA3s')
print(f'  New in 2024 (crossed threshold): {len(new_crisis)} communities')
print(f'  Resolved (dropped below 1.0):    {len(resolved)} communities')
print()
print('Communities that became crisis zones in 2024:')
new_detail = m24[m24['sa3_name'].isin(new_crisis)].sort_values('waitlist_pressure', ascending=False)
print(new_detail[['sa3_name','state','mmm_code','waitlist_pressure']].round(3).to_string(index=False))
print()
print('What next: Without structural intervention — new bed construction, rezoning,'
      ' workforce expansion — the 2025 data will show more communities crossing the line.')

Communities in deficit (pressure > 1.0):
  2023: 48 SA3s
  2024: 58 SA3s
  New in 2024 (crossed threshold): 14 communities
  Resolved (dropped below 1.0):    4 communities

Communities that became crisis zones in 2024:
                      sa3_name state mmm_code  waitlist_pressure
     Daly - Tiwi - West Arnhem    NT      NaN              1.220
            Wheat Belt - South    WA      MM5              1.139
             Campbelltown (SA)    SA      MM1              1.122
             Murray and Mallee    SA      MM5              1.095
                       Sunbury   VIC      MM1              1.068
                   Onkaparinga    SA      MM1              1.065
               Darebin - South   VIC      MM1              1.063
        Gippsland - South West   VIC      MM4              1.063
                      Brighton   TAS      MM2              1.062
   Tablelands (East) - Kuranda   QLD      MM4              1.031
                 Charles Sturt    SA      MM1              1.022
 

### What this means for the trajectory

In a single year, **14 new communities crossed the 1.0 threshold** — meaning demand now structurally exceeds supply in places it did not a year ago. Only 4 communities resolved their deficit. The net change is +10 crisis communities in 12 months.

The 14 new entrants are geographically spread — NT, WA, SA, VIC, TAS, QLD — and span all remoteness bands from MM1 cities (Sunbury VIC, Campbelltown SA, Dandenong VIC) to MM4-5 rural areas (Gippsland South West, Wheat Belt South). This is not a regional problem migrating — it is a national pattern expanding.

**The policy implication is asymmetric urgency.** A community at 0.9 pressure today will cross the threshold within 12–18 months at current growth rates. Proactive infrastructure investment at that point costs a fraction of crisis response once pressure reaches 2.0+. The 14 communities that crossed in 2024 were — in 2023 — exactly the kind of early-warning cases that targeted funding could have prevented.

## Call to Action: What This Means for Each Audience

---

### 👨‍👩‍👧 For Families Planning Aged Care

If your region appears in this list, **start the ACAT assessment process now, not when crisis hits.** The waitlist in regions like Noosa Hinterland (2.83) or Sunshine Coast Hinterland (2.62) means that by the time a relative is assessed as needing residential care urgently, the queue is already years long.

**Practical step:** Use this chapter's table to check your region's pressure score. If it is above 1.5, build a transition plan that includes relocation options.

---

### 🏛️ For Health System Strategists and Policymakers

National bed targets miss the point. The crisis is geographic. **58 specific SA3 regions are in deficit** — and the deficit is measurable, comparable, and actionable.

The 10 new communities that crossed the 1.0 threshold in a single year (2023→2024) represent a leading indicator, not a lagging one. Infrastructure funding directed at these communities before they reach 2.0 pressure is categorically more efficient than crisis response.

**Practical step:** Use the `waitlist_pressure` ranking as the primary criterion for aged care capital allocation in the next infrastructure round.

---

### 👷 For Workers and Businesses Entering the Sector

115 SA3 regions lost residential facilities between 2019 and 2024. These are not markets that are saturated — they are markets that have been **abandoned by operators who found them uneconomical**, leaving behind documented, growing, unmet demand.

For NFP operators and government programs, these abandoned communities represent the highest-impact deployment of new capacity. For workers, they represent the strongest job security in the sector — places where demand will not decline.

**Practical step:** Cross-reference `supply_change < 0` with `waitlist_pressure > 1.0` in `master_sa3.csv` to identify the highest-opportunity markets.

---

## Data Limitation Note

Indigenous and NESB demographic breakdown is only available at ACPR level (73 regions) — it cannot be joined to SA3 (358 regions). This chapter does not include demographic equity analysis at the SA3 level. If demographic equity is a priority, a separate ACPR-level analysis is required, clearly labelled as a different geographic unit.